## Practice with Zarr

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join("Climsim", "diffusion-climsim", "diffusionsim")))
import diffusers
import diffusionsim as diff
import diffusionsim.training_utils as tru
from diffusionsim import mydatasets as data
from diffusionsim import climsim_utils as cut
path = lambda fname : os.path.join(os.path.expanduser("~/Climsim/diffusion-climsim/"), fname)

In [2]:
from virtualizarr import open_virtual_dataset
import zarr
import kerchunk
import icechunk
import xarray as xr
import fsspec
import dask
import json
import numpy as np

### Parallelizing With Dask

In [25]:
from dask_gateway import Gateway
from dask.distributed import Client

In [26]:
client = Client(n_workers=20)

2025-03-11 01:53:39,456 - bokeh.server.protocol_handler - ERROR - error handling message
 message: Message 'PULL-DOC-REQ' content: {} 
 error: SerializationError("can't serialize <class 'function'>")
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/bokeh/server/protocol_handler.py", line 94, in handle
    work = await handler(message, connection)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/bokeh/server/session.py", line 94, in _needs_document_lock_wrapper
    result = func(self, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/python3.12/site-packages/bokeh/server/session.py", line 257, in _handle_pull
    return connection.protocol.create('PULL-DOC-REPLY', message.header['msgid'], self.document)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/srv/conda/envs/notebook/lib/pyt

In [10]:
options = gateway.cluster_options()
# options.worker_memory = 16

options

In [4]:
gateway = Gateway()

In [5]:
gateway.list_clusters()

[]

In [6]:
cluster = gateway.new_cluster()

In [7]:
options = gateway.cluster_options()

In [12]:
#options.worker_cores
cluster.scale(20)
client = Client(cluster)

In [13]:
client

Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/prod.0607678a0bde49bc970adcba72c93bb1/status,


In [14]:
dconfig = tru.DataConfig()
dconfig.source = "huggingface"
dconfig.climsim_type = "low-res-expanded"

n = cut.expand_ds_name("low-res")
grid_url = f"https://huggingface.co/datasets/LEAP/{n}/resolve/main/{n}_grid-info.nc"
grid_info = cut.read_url(grid_url, copy_to_local=True)

dutils = cut.setup_data_utils(dconfig.climsim_type, dconfig.source, dconfig.data_vars, use_tendencies=dconfig.use_tendencies)



In [22]:
@dask.delayed
def fetch_file_delayed(fname, dutils=dutils):
    path = os.path.join(dutils.data_path, fname)
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = open_virtual_dataset(path)
    time = dutils.parse_time(fname)
    ds = dutils.add_time(ds, time)
    return ds

def fetch_file(fname, dutils=dutils):
    path = os.path.join(dutils.data_path, fname)
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = open_virtual_dataset(path)
    time = dutils.parse_time(fname)
    ds = dutils.add_time(ds, time)
    return ds

In [16]:
dutils.set_filelist_using_hfhub("train", 6, 5, 1)

In [17]:
filelist = dutils.get_filelist("train")
print(f"{len(filelist)} files, from {filelist[0]} to {filelist[-1]}")

2232 files, from 0006-05/E3SM-MMF.mlexpand.0006-05-01-00000.nc to 0006-05/E3SM-MMF.mlexpand.0006-05-31-85200.nc


In [28]:
delayed_list = [fetch_file_delayed(fname) for fname in filelist[::200]]

In [29]:
delayed_list

[Delayed('fetch_file_delayed-7888677f-7759-495d-a17d-492b63d61747'),
 Delayed('fetch_file_delayed-bc362e8e-3674-4a1b-9ede-075e71eeaef3'),
 Delayed('fetch_file_delayed-fec26915-fb79-47f8-80c5-99ab64a43a80'),
 Delayed('fetch_file_delayed-6f022f52-23ea-48b5-89b3-0d46b308e30d'),
 Delayed('fetch_file_delayed-fd0730e7-9c9d-47d1-81b9-d1060b38ddb8'),
 Delayed('fetch_file_delayed-4f666e86-d394-448b-a4af-eb28ee336371'),
 Delayed('fetch_file_delayed-1144d322-367e-40a1-99b5-7beb7e4dbdbf'),
 Delayed('fetch_file_delayed-4816b315-ed4f-4db4-bd38-28139cfd6986'),
 Delayed('fetch_file_delayed-424ef097-c146-4887-80dd-d572d31da403'),
 Delayed('fetch_file_delayed-253b2849-9ab0-40f2-ae06-82367495e189'),
 Delayed('fetch_file_delayed-92f8c4c3-aa45-4a71-8c13-7c3f1807e1a0'),
 Delayed('fetch_file_delayed-206a2646-0b14-4e76-9104-5a07f6448ca2')]

In [24]:
virtual_datasets = [fetch_file(fname) for fname in filelist[::600]]

In [30]:
loaded_filelist = client.compute(delayed_list)

In [33]:
loaded_filelist

[<Future: pending, key: fetch_file_delayed-7888677f-7759-495d-a17d-492b63d61747>,
 <Future: pending, key: fetch_file_delayed-bc362e8e-3674-4a1b-9ede-075e71eeaef3>,
 <Future: pending, key: fetch_file_delayed-fec26915-fb79-47f8-80c5-99ab64a43a80>,
 <Future: pending, key: fetch_file_delayed-6f022f52-23ea-48b5-89b3-0d46b308e30d>,
 <Future: pending, key: fetch_file_delayed-fd0730e7-9c9d-47d1-81b9-d1060b38ddb8>,
 <Future: pending, key: fetch_file_delayed-4f666e86-d394-448b-a4af-eb28ee336371>,
 <Future: pending, key: fetch_file_delayed-1144d322-367e-40a1-99b5-7beb7e4dbdbf>,
 <Future: pending, key: fetch_file_delayed-4816b315-ed4f-4db4-bd38-28139cfd6986>,
 <Future: pending, key: fetch_file_delayed-424ef097-c146-4887-80dd-d572d31da403>,
 <Future: pending, key: fetch_file_delayed-253b2849-9ab0-40f2-ae06-82367495e189>,
 <Future: pending, key: fetch_file_delayed-92f8c4c3-aa45-4a71-8c13-7c3f1807e1a0>,
 <Future: pending, key: fetch_file_delayed-206a2646-0b14-4e76-9104-5a07f6448ca2>]

In [32]:
monthly_input_vds = client.gather(loaded_filelist)

KeyboardInterrupt: 

In [ ]:
monthly_input_vds

## Testing Virtualizarr thru HF

In [24]:
dirs = '/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/hf_manifests/mlo'
storage = icechunk.local_filesystem_storage(dirs)
repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")

In [25]:
with session.allow_pickling():
    loaded_hfds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks={})

FileNotFoundError: Unable to find group: <icechunk.store.IcechunkStore object at 0x1554321fb2c0>

In [6]:
base_dir = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/hf_manifests"

In [7]:
storage = icechunk.local_filesystem_storage(base_dir + "/mlo")
repo = icechunk.Repository.open(storage)

IcechunkError:   x unknown storage error: Read-only file system (os error 30)
  | 
  | context:
  | 
  | 
  `-> unknown storage error: Read-only file system (os error 30)


In [21]:
delayed_list

[Delayed('fetch_file-ef5b52ec-e8a9-43b4-bae4-2a0870d2218b'),
 Delayed('fetch_file-a40f3949-560f-4009-a66d-bcd7a5460527'),
 Delayed('fetch_file-732d8eef-8513-44c8-82bf-4cf683ebe5bf'),
 Delayed('fetch_file-dbb6987e-f9db-41c5-a4ce-849c81d720e6'),
 Delayed('fetch_file-46b338b0-fe17-4f49-86bc-9cd9b61e3a65'),
 Delayed('fetch_file-8c8b8659-368d-412a-87a3-42f72144a85c'),
 Delayed('fetch_file-be4696ac-22c5-4bf3-a0f5-e1ddee1af277'),
 Delayed('fetch_file-13f83ceb-eff4-47a0-abfb-e31d9e86b7cb'),
 Delayed('fetch_file-2728042b-d1eb-4349-ad14-f781f7fae600'),
 Delayed('fetch_file-2da56fce-e056-4a1d-8923-610291455b22'),
 Delayed('fetch_file-ab0507f1-e189-46c8-8735-58d2d2d568d3'),
 Delayed('fetch_file-b6e614e0-059c-46ca-8f12-8396b54e823a')]

## Setting up Virtualizarr thru to Huggingface

## Creating

In [3]:
def iterate_months():
    for year in range(1,10):
        for month in range(1,13):
            if(year == 1 and month == 1):
                continue
            if(year == 9 and month > 1):
                break
            yield(year, month)

In [8]:
for year, month in iterate_months():
    print(year, month)

In [4]:
dconfig = tru.DataConfig()
dconfig.source = "huggingface"
dconfig.climsim_type = "low-res-expanded"



In [5]:
n = cut.expand_ds_name("low-res")
grid_url = f"https://huggingface.co/datasets/LEAP/{n}/resolve/main/{n}_grid-info.nc"
grid_info = cut.read_url(grid_url, copy_to_local=True)

ERROR 1: PROJ: proj_create_from_database: Open of /srv/conda/envs/notebook/share/proj failed


In [6]:
dutils = cut.setup_data_utils(dconfig.climsim_type, dconfig.source, dconfig.data_vars, use_tendencies=dconfig.use_tendencies)



In [8]:
dutils.set_filelist_using_hfhub("train", 4, 5, 1)
filelist = dutils.get_filelist("train")[::100]

In [9]:
print(len(dutils.get_filelist("train")))

dutils.get_filelist("train")[-5:]

2232


['0004-05/E3SM-MMF.mlexpand.0004-05-31-80400.nc',
 '0004-05/E3SM-MMF.mlexpand.0004-05-31-81600.nc',
 '0004-05/E3SM-MMF.mlexpand.0004-05-31-82800.nc',
 '0004-05/E3SM-MMF.mlexpand.0004-05-31-84000.nc',
 '0004-05/E3SM-MMF.mlexpand.0004-05-31-85200.nc']

In [10]:
virtual_datasets = []
for fname in filelist:
    vds = dutils.get_xrdata(fname, virtual=True)
    virtual_datasets.append(vds)

In [11]:
virtual_ds = xr.combine_nested(virtual_datasets, concat_dim=['time'])
virtual_ds

<xarray.Dataset> Size: 138MB
Dimensions:                (time: 23, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) object 184B 0004-05-01 00:00:00 ... 0004-05...
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    ymd                    (time) int32 92B ManifestArray<shape=(23,), dtype=...
    tod                    (time) int32 92B ManifestArray<shape=(23,), dtype=...
    cam_in_ALDIF           (time, ncol) float64 71kB ManifestArray<shape=(23,...
    cam_in_ALDIR           (time, ncol) float64 71kB ManifestArray<shape=(23,...
    cam_in_ASDIF           (time, ncol) float64 71kB ManifestArray<shape=(23,...
    cam_in_ASDIR           (time, ncol) float64 71kB ManifestArray<shape=(23,...
    ...                     ...
    tm_pbuf_COSZRS         (time, ncol) float64 71kB ManifestArray<shape=(23,...
    lat                    (time, ncol) float64 71kB ManifestArray<shape=(23,...
    lon                    (time, ncol) float64 71kB ManifestArray<shape=(23,...
    clat                   (time, ncol) float64 71kB ManifestArray<shape=(23,...
    slat                   (time, ncol) float64 71kB ManifestArray<shape=(23,...
    icol                   (time, ncol) float64 71kB ManifestArray<shape=(23,...

In [12]:
dutils.mlivar

'mlexpand'

In [13]:
storage = icechunk.local_filesystem_storage("/mnt/home/ssa2206/Climsim/test_hf_manifest")
repo = icechunk.Repository.create(storage)

In [14]:
session = repo.writable_session("main")
virtual_ds.virtualize.to_icechunk(session.store)

In [16]:
session.commit("My first virtual store!")

'X509ZFXSKZAAZAKMSQ5G'

### Appending more data on existing store

In [17]:
vds2 = dutils.get_xrdata(dutils.get_filelist("train")[5], virtual=True)
vds2

<xarray.Dataset> Size: 6MB
Dimensions:                (time: 1, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) object 8B 0004-05-01 01:40:00
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    ymd                    (time) int32 4B ManifestArray<shape=(1,), dtype=in...
    tod                    (time) int32 4B ManifestArray<shape=(1,), dtype=in...
    cam_in_ALDIF           (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    cam_in_ALDIR           (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    cam_in_ASDIF           (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    cam_in_ASDIR           (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    ...                     ...
    tm_pbuf_COSZRS         (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    lat                    (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    lon                    (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    clat                   (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    slat                   (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
    icol                   (time, ncol) float64 3kB ManifestArray<shape=(1, 3...
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [18]:
session = repo.writable_session("main")
vds2.virtualize.to_icechunk(session.store, append_dim='time')
session.commit("Appended another time dimension")

'SK7379E9W5HK3WB40HKG'

## Loading Existing Store and Examining 

In [10]:
manifest_dir = "/mnt/home/ssa2206/Climsim/test_hf_manifest"
manifest_dir = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/aggregate_manifest"
manifest_dir = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/hf_manifests/mlexpand"

In [15]:
storage = icechunk.local_filesystem_storage(manifest_dir)
repo2 = icechunk.Repository.open(storage)
session = repo2.writable_session("main")

In [16]:
with session.allow_pickling():
    loaded_ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks={})

In [17]:
loaded_ds.sizes#.time[-5:] #+ loaded_ds.time[-5:]

Frozen({'time': 19654, 'ncol': 384, 'lev': 60})

In [27]:
loaded_ds.nbytes / 1e9


loaded_ds.cam_in_ALDIF.nbytes_loaded

AttributeError: 'DataArray' object has no attribute 'nbytes_loaded'

In [18]:
for snapshot in repo2.ancestry(branch="main"):
    print(snapshot.message)

Appended 1-10 for inputs
Appended 1-9 for inputs
Appended 1-8 for inputs
Appended 1-7 for inputs
Appended 1-6 for inputs
Appended 1-5 for inputs
Appended 1-4 for inputs
Appended 1-3 for inputs
Appended 1-2 for inputs
Repository initialized


In [44]:
%%time
url = "https://huggingface.co/datasets/LEAP/ClimSim_low-res-expanded/resolve/main/train/0004-05/E3SM-MMF.mlexpand.0004-05-01-00000.nc"
ds = open_virtual_dataset(url)


CPU times: user 405 ms, sys: 32.9 ms, total: 438 ms
Wall time: 1.7 s


In [49]:
ds.virtualize.to_kerchunk("vzarr.json", format='json')

In [46]:
ds.virtualize.to_kerchunk("combined2.parquet", format='parquet')

In [47]:
cds = xr.open_dataset('combined2.parquet', engine="kerchunk", chunks={})

In [50]:
%%time
cds.load()

CPU times: user 396 ms, sys: 163 ms, total: 559 ms
Wall time: 1.47 s


<xarray.Dataset> Size: 6MB
Dimensions:                (ncol: 384, lev: 60)
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    cam_in_ALDIF           (ncol) float64 3kB 1.0 1.0 1.0 ... 0.0899 0.09562
    cam_in_ALDIR           (ncol) float64 3kB 1.0 1.0 1.0 ... 0.05562 0.06306
    cam_in_ASDIF           (ncol) float64 3kB 1.0 1.0 1.0 ... 0.08266 0.05857
    cam_in_ASDIR           (ncol) float64 3kB 1.0 1.0 1.0 ... 0.05416 0.03521
    cam_in_ICEFRAC         (ncol) float64 3kB 0.0 0.0 0.0 ... 0.04101 0.0
    cam_in_LANDFRAC        (ncol) float64 3kB 0.0 0.0 0.2982 ... 0.1417 0.2428
    ...                     ...
    tm_state_u             (lev, ncol) float64 184kB 51.9 19.36 ... -4.957 6.199
    tm_state_u_dyn         (lev, ncol) float64 184kB -0.0002776 ... 7.043e-05
    tm_state_u_prvphy      (lev, ncol) float64 184kB 0.0 0.0 ... 4.532e-05
    tm_state_v             (lev, ncol) float64 184kB 0.6186 -19.15 ... -0.731
    tod                    float64 8B 0.0
    ymd                    float64 8B 4.05e+04
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [ ]:
cds2 = xr.open_dataset("vzarr.json", engine="kerchunk", chunks={})

In [ ]:
#virtual_ds.virtualize.to_kerchunk(path("combined.json"), format='json')
virtual_ds.virtualize.to_kerchunk(path("combined2.parquet"), format='parquet')

combined_ds = xr.open_dataset(path('combined2.parquet'), engine="kerchunk", chunks={})

In [40]:
get_xrdata(dutils, filelist[0], virtual=True)

https://huggingface.co/datasets/LEAP/ClimSim_low-res-expanded/resolve/main/train/0004-05/E3SM-MMF.mlexpand.0004-05-01-00000.nc


<xarray.Dataset> Size: 6MB
Dimensions:                (time: 1, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) object 8B 0004-05-01 00:00:00
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    ymd                    (time) int32 4B 40501
    tod                    (time) int32 4B 0
    cam_in_ALDIF           (time, ncol) float64 3kB 1.0 1.0 ... 0.0899 0.09562
    cam_in_ALDIR           (time, ncol) float64 3kB 1.0 1.0 ... 0.05562 0.06306
    cam_in_ASDIF           (time, ncol) float64 3kB 1.0 1.0 ... 0.08266 0.05857
    cam_in_ASDIR           (time, ncol) float64 3kB 1.0 1.0 ... 0.05416 0.03521
    ...                     ...
    tm_pbuf_COSZRS         (time, ncol) float64 3kB 0.0 0.0 ... 0.7324 0.6624
    lat                    (time, ncol) float64 3kB -32.59 -35.99 ... 40.39
    lon                    (time, ncol) float64 3kB 320.3 331.5 ... 146.7 135.0
    clat                   (time, ncol) float64 3kB 0.8426 0.8091 ... 0.7616
    slat                   (time, ncol) float64 3kB -0.5386 -0.5877 ... 0.648
    icol                   (time, ncol) float64 3kB 1.0 2.0 3.0 ... 383.0 384.0
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

## Setting up on Empire Disk

In [40]:
dconfig = tru.DataConfig()
dconfig.source = "local-vzarr"
dconfig.climsim_type = "low-res-expanded"
dconfig.data_dir = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/train"


In [39]:
n = cut.expand_ds_name("low-res")
grid_url = f"https://huggingface.co/datasets/LEAP/{n}/resolve/main/{n}_grid-info.nc"

kwargs = {
    'base_dir' : "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/train",
    'grid_info' : cut.read_url(grid_url, copy_to_local=True)
}

dutils = cut.setup_data_utils(dconfig.climsim_type, dconfig.source, dconfig.data_vars, use_tendencies=dconfig.use_tendencies, **kwargs)

start = cut.tocft(2, 1, 1)
stop = cut.tocft(2, 1, 15)

dutils.set_filelist_using_intervals("train", start, stop, 20*3*6)

In [41]:
fpath = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/aggregate_manifest"

storage = icechunk.local_filesystem_storage(fpath)
repo = icechunk.Repository.open(storage)
session = repo.writable_session("main")

ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, chunks={})

In [45]:
dso = cut.add_space(ds[dutils.target_vars], ds_grid=dutils.grid_info)

In [48]:
dsets, indices = data.train_test_split(dso, dso, dconfig.train_test_split)

In [53]:
dataset = data.XBatchDataset(dsets[0][1], dconfig, log=True)

In [131]:
start = cut.tocft(4, 6, 1)
stop = cut.tocft(4, 7, 1)
dutils.set_filelist_using_intervals("train", start, stop, 20*3*6)
dutils.get_filelist("train")

['0004-06/E3SM-MMF.mlexpand.0004-06-01-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-64800.nc',


In [12]:
filelist = dutils.get_filelist("train")
len(filelist)

56

In [126]:
virtual_datasets = []
times = []
for i, fname in enumerate(filelist[:10]):
    fpath = os.path.join(dutils.data_path, fname)
    ds = open_virtual_dataset(fpath)
    time = dutils.parse_time(fname)
    times.append(time)
    ds = ds.expand_dims(time=[i+1])
    #ds["time"] = (["time"], [i])
    virtual_datasets.append(ds)


In [137]:
np.load(path("times.npy"), allow_pickle=True)

array([cftime.DatetimeNoLeap(2, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 6, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 18, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 6, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 18, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 3, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 3, 6, 0, 0, 0, has_year_zero=True)],
      dtype=object)

In [135]:
np.save(path("times.npy"), times, allow_pickle=True)

In [86]:
virtual_ds = xr.combine_nested(virtual_datasets, concat_dim=['time'])
virtual_ds

<xarray.Dataset> Size: 60MB
Dimensions:                (time: 10, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) int64 80B 1 2 3 4 5 6 7 8 9 10
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    ymd                    (time) int32 40B ManifestArray<shape=(10,), dtype=...
    tod                    (time) int32 40B ManifestArray<shape=(10,), dtype=...
    cam_in_ALDIF           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ALDIR           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ASDIF           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ASDIR           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    ...                     ...
    tm_pbuf_COSZRS         (time, ncol) float64 31kB ManifestArray<shape=(10,...
    lat                    (time, ncol) float64 31kB ManifestArray<shape=(10,...
    lon                    (time, ncol) float64 31kB ManifestArray<shape=(10,...
    clat                   (time, ncol) float64 31kB ManifestArray<shape=(10,...
    slat                   (time, ncol) float64 31kB ManifestArray<shape=(10,...
    icol                   (time, ncol) float64 31kB ManifestArray<shape=(10,...

In [88]:
#virtual_ds.virtualize.to_kerchunk(path("combined.json"), format='json')
virtual_ds.virtualize.to_kerchunk(path("combined2.parquet"), format='parquet')

In [89]:
combined_ds = xr.open_dataset(path('combined2.parquet'), engine="kerchunk", chunks={})

In [ ]:
ds = xr.open_dataset('combined.json', engine='kerchunk', chunks={}

In [96]:
dsi

<xarray.Dataset> Size: 4MB
Dimensions:      (time: 10, lev: 60, ncol: 384)
Coordinates:
  * time         (time) object 80B 0002-01-01 00:00:00 ... 0002-01-03 06:00:00
Dimensions without coordinates: lev, ncol
Data variables:
    state_t      (time, lev, ncol) float64 2MB -inf -inf inf ... -inf -inf -inf
    state_q0001  (time, lev, ncol) float64 2MB -inf -inf -inf ... -inf -inf -inf
    state_ps     (time, ncol) float64 31kB inf inf inf inf ... -inf -inf inf inf
    pbuf_SOLIN   (time, ncol) float64 31kB -inf -inf -inf -inf ... inf -inf -inf
    pbuf_LHFLX   (time, ncol) float64 31kB -inf -inf -inf -inf ... -inf inf inf
    pbuf_SHFLX   (time, ncol) float64 31kB -inf -inf -inf -inf ... inf inf inf

In [99]:
ds_inputs, ds_targets = [], []
for i, file in enumerate(filelist):
    ds_input = dutils.get_input(file)
    ds_target = dutils.get_target(file)
    ds_inputs.append(ds_input)
    ds_targets.append(ds_target)

ds_inputs = xr.concat(ds_inputs, dim='time')
ds_targets = xr.concat(ds_targets, dim='time')


In [104]:
ds_inputs

<xarray.Dataset> Size: 21MB
Dimensions:      (time: 56, lev: 60, ncol: 384)
Coordinates:
  * time         (time) object 448B 0002-01-01 00:00:00 ... 0002-01-14 18:00:00
Dimensions without coordinates: lev, ncol
Data variables:
    state_t      (time, lev, ncol) float64 10MB 212.6 209.0 ... 267.4 268.7
    state_q0001  (time, lev, ncol) float64 10MB 1.046e-06 1.046e-06 ... 0.00189
    state_ps     (time, ncol) float64 172kB 1.007e+05 1.016e+05 ... 1.004e+05
    pbuf_SOLIN   (time, ncol) float64 172kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    pbuf_LHFLX   (time, ncol) float64 172kB 41.66 31.69 53.02 ... 82.58 100.7
    pbuf_SHFLX   (time, ncol) float64 172kB 3.204 4.14 8.524 ... 93.91 86.44
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [109]:
ds_inputs['state_t'].data

array([[[212.5978174 , 208.96261498, 219.0162239 , ..., 216.18261307,
         217.87367276, 223.77638051],
        [218.3212459 , 216.91067342, 226.43448218, ..., 219.37087145,
         237.19641764, 237.76134192],
        [231.7245395 , 232.51944625, 232.54869655, ..., 231.19627076,
         238.42716583, 238.35975779],
        ...,
        [291.50802812, 287.98873335, 294.67627256, ..., 255.0645094 ,
         272.16110837, 272.37923022],
        [292.48501245, 289.10676856, 295.1791328 , ..., 256.02903722,
         272.9716271 , 273.52246283],
        [293.64563862, 290.28940923, 296.05201769, ..., 257.02499625,
         274.0153692 , 274.7273004 ]],

       [[210.7304669 , 210.0221052 , 215.13246685, ..., 219.19644466,
         224.72313981, 226.18673286],
        [218.4391942 , 226.19061597, 217.44592552, ..., 224.14801855,
         237.62795576, 232.95331796],
        [230.16199482, 234.45166202, 230.65427327, ..., 234.20125841,
         238.22917903, 239.75267657],
        ...,


In [112]:
for var in dutils.input_vars:
    print(var)
    print((ds_inputs[var].data[:10] == combined_ds[var].data).all())


state_t
True
state_q0001
True
state_ps
True
pbuf_SOLIN
True
pbuf_LHFLX
True
pbuf_SHFLX
True


In [110]:
combined_ds

<xarray.Dataset> Size: 60MB
Dimensions:                (time: 10, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) float64 80B 1.0 2.0 3.0 4.0 ... 8.0 9.0 10.0
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    cam_in_ALDIF           (time, ncol) float64 31kB 1.0 1.0 ... 0.09448 0.09341
    cam_in_ALDIR           (time, ncol) float64 31kB 1.0 1.0 ... 0.3732 0.2052
    cam_in_ASDIF           (time, ncol) float64 31kB 1.0 1.0 ... 0.09298 0.06892
    cam_in_ASDIR           (time, ncol) float64 31kB 1.0 1.0 ... 0.3573 0.1778
    cam_in_ICEFRAC         (time, ncol) float64 31kB 0.0 0.0 0.0 ... 0.0 0.0
    cam_in_LANDFRAC        (time, ncol) float64 31kB 0.0 0.0 ... 0.1417 0.2428
    ...                     ...
    tm_state_u             (time, lev, ncol) float64 2MB -41.94 -58.75 ... 6.64
    tm_state_u_dyn         (time, lev, ncol) float64 2MB 0.001111 ... -0.0001621
    tm_state_u_prvphy      (time, lev, ncol) float64 2MB 0.0 0.0 ... 0.0001139
    tm_state_v             (time, lev, ncol) float64 2MB -9.322 ... -3.942
    tod                    (time) float64 80B 0.0 2.16e+04 ... 0.0 2.16e+04
    ymd                    (time) float64 80B 2.01e+04 2.01e+04 ... 2.01e+04

##### Debugging

In [ ]:
ref = json.load(open(os.path.expanduser("~/diffusion-climsim/combined.json"), 'r'))

### Working with actual zarrs

[Zarr Array Docs](https://zarr.readthedocs.io/en/latest/user-guide/arrays.html)

In [ ]:
mapper = fsspec.get_mapper("reference://", 
                           fo=path("combined.json"), 
                           target_protocol="file")



zgroup = zarr.open_group(mapper, mode="r+")

In [52]:
store = zarr.storage.MemoryStore()
z = zarr.create_array(store=store, shape=(1000, 1000), chunks=(100, 100), dtype='int32')
z

<Array memory://23451371698944 shape=(1000, 1000) dtype=int32>

In [56]:
z[0, :] = np.arange(1000)
z[:, 0] = np.arange(1000)

In [57]:
z[:]

array([[  0,   1,   2, ..., 997, 998, 999],
       [  1,   0,   0, ...,   0,   0,   0],
       [  2,   0,   0, ...,   0,   0,   0],
       ...,
       [997,   0,   0, ...,   0,   0,   0],
       [998,   0,   0, ...,   0,   0,   0],
       [999,   0,   0, ...,   0,   0,   0]], dtype=int32)